# 05 - Optional: from 2D to 3D

In the project the network sees one slice at a time. A CT scan is a volume: here we train a network that
segments the lungs in the **whole 3D scan** at once.

The good news is that almost nothing changes. The steps are the same as in the project, and for each step
we only point out what is different in 3D. Keep the project notebook open: you will copy code from it.

To keep the training fast we use a reduced version of the scans, with voxels of 2.5 mm.

As in the project: replace every `...`, use the variable names of the instructions, and compare your result
with "You should see" before going on.

# STEP 0: Setup

Run the cells. The Accelerator must be **GPU T4 x2**.

In [ ]:
!pip install -q monai

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from monai.data import DataLoader, Dataset
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    Lambdad,
    LoadImaged,
    Resized,
    ScaleIntensityRanged,
)
from monai.utils import set_determinism

In [ ]:
DATA_DIR = Path("/kaggle/input/luna16-lung-slices") / "volumes_lowres"

set_determinism(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# STEP 1: The list of files

The same as STEP 1 and STEP 3 of the project, with the new `DATA_DIR`.

1. Create `dataset_files`: one dictionary `{"image": ..., "label": ...}` per patient.
2. Divide it in `train_files` (80%) and `val_files` (20%) with `train_test_split` and `random_state=42`.
3. Print the number of patients in the two groups.

In [ ]:
# 1. One dictionary per patient (hint: STEP 1 of the project)
dataset_files = ...

# 2. Training and validation
train_files, val_files = ...

# 3. Size of the two groups
print(...)

**You should see** two numbers: 80% and 20% of the scans available.

# STEP 2: Look at a volume

**What changes in 3D:** the array has three dimensions (x, y, z), so to plot it we must choose a slice.

1. Load the image and the mask of the first training patient with `nibabel`, in `volume` and `mask`.
2. Print the shape of `volume` and the voxel size from the header.
3. Run the plot: it shows the middle axial slice and the middle coronal slice, with the mask.

In [ ]:
# 1. Load image and mask (hint: nib.load(train_files[0]["image"]))
volume_nii = ...
volume = volume_nii.get_fdata()
mask = ...

# 2. Shape and voxel size (hint: volume_nii.header.get_zooms())
print(...)
print(...)

# 3. Axial and coronal view
x, y, z = volume.shape

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Axial")
plt.imshow(volume[:, :, z // 2].T, cmap="gray", vmin=-1000, vmax=400)
plt.imshow(np.ma.masked_equal(mask[:, :, z // 2], 0).T, cmap="autumn", alpha=0.3)
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("Coronal")
plt.imshow(volume[:, y // 2, :].T, cmap="gray", vmin=-1000, vmax=400, origin="lower")
plt.imshow(np.ma.masked_equal(mask[:, y // 2, :], 0).T, cmap="autumn", alpha=0.3, origin="lower")
plt.axis("off")
plt.show()

**You should see** a shape with three numbers (about 150 x 150 x 120), a voxel size of `(2.5, 2.5, 2.5)` and
two views of the chest with the lungs coloured.

# STEP 3: Transforms

**What changes in 3D:** only the resize. The size has three numbers, and the interpolation of the image is
called `"trilinear"` instead of `"bilinear"`. All the other transforms work in 2D and in 3D without changes.

The scans have different shapes. We resize all of them to 96 x 96 x 96.

1. Complete `transforms` with the same five steps of STEP 4 of the project, with the 3D resize.

We do not use augmentation here, to keep things simple.

In [ ]:
IMG_SIZE = 96

def keep_lungs(label):
    return ((label == 1) | (label == 2)).float()

# 1. The five steps (hint: copy val_transforms from the project, then change spatial_size and mode)
transforms = Compose([
    ...
])

check = transforms(train_files[0])
print("image:", check["image"].shape)
print("label:", check["label"].shape, "values", np.unique(check["label"]))

**You should see**

```
image: torch.Size([1, 96, 96, 96])
label: torch.Size([1, 96, 96, 96]) values [0. 1.]
```

# STEP 4: Dataset and DataLoader

**What changes in 3D:** a volume uses much more memory than a slice, so the batch is smaller: 2 volumes.

This time we use the `Dataset` class of MONAI, which does the same as the `SegmentationDataset` we wrote
in the project.

1. Create `train_ds` and `val_ds` with `Dataset(data=..., transform=...)`.
2. Create `train_loader` (`batch_size=2`, `shuffle=True`, `num_workers=2`) and `val_loader`
   (`batch_size=2`, `shuffle=False`, `num_workers=2`).
3. Take the first batch of `train_loader` and print the shape of `batch["image"]`.

In [ ]:
# 1. Datasets
train_ds = ...
val_ds = ...

# 2. Loaders
train_loader = ...
val_loader = ...

# 3. First batch
batch = next(iter(train_loader))
print(...)

**You should see** `torch.Size([2, 1, 96, 96, 96])`: 2 volumes, 1 channel, 96 x 96 x 96 voxels.

# STEP 5: The model

**What changes in 3D:** one argument.

1. Create `model`: the same `UNet` of the project, but for 3D images. Use `channels=(16, 32, 64, 128)` and
   `strides=(2, 2, 2)`, a smaller network than in the project.
2. Create `loss_function`, `optimizer` and `dice_metric` exactly as in STEP 7 of the project.
3. Pass `batch["image"]` through the model and print the shape of the output.

In [ ]:
# 1. The model
model = ...

# 2. Loss, optimizer, metric
loss_function = ...
optimizer = ...
dice_metric = ...

# 3. Output shape
outputs = ...
print(...)

**You should see** `torch.Size([2, 2, 96, 96, 96])`.

# STEP 6: Training

**What changes in 3D:** nothing. The loop does not know whether the tensors are 2D or 3D.

1. Copy here the training loop of STEP 9 of the project.
2. Change the name of the saved file to `"lung_segmenter_3d.pth"`.
3. Set `MAX_EPOCHS = 30` and run.

In [ ]:
# 1. - 3. The training loop of the project, with the new file name and MAX_EPOCHS = 30
...

**You should see** one line per epoch, with the validation Dice going up as in the project.

# STEP 7: Look at the predictions

1. Load the best weights and put the model in evaluation mode.
2. Compute `preds` for the first batch of `val_loader`, as in STEP 11 of the project.
3. Run the plot: axial and coronal view of the first patient, reference mask and prediction.

In [ ]:
# 1. Best weights, evaluation mode
...

# 2. Predictions on the first validation batch
batch = ...
with torch.no_grad():
    outputs = ...
    preds = ...

# 3. First patient of the batch: reference mask (top) and prediction (bottom)
image = batch["image"][0, 0]
label = batch["label"][0, 0]
pred = preds[0, 0].cpu()
m = IMG_SIZE // 2

plt.figure(figsize=(10, 10))
for row, (name, seg) in enumerate([("Reference mask", label), ("Prediction", pred)]):
    plt.subplot(2, 2, 2 * row + 1)
    plt.title(f"{name} - axial")
    plt.imshow(image[:, :, m].T, cmap="gray")
    plt.imshow(np.ma.masked_equal(seg[:, :, m], 0).T, cmap="autumn", alpha=0.4)
    plt.axis("off")
    plt.subplot(2, 2, 2 * row + 2)
    plt.title(f"{name} - coronal")
    plt.imshow(image[:, m, :].T, cmap="gray", origin="lower")
    plt.imshow(np.ma.masked_equal(seg[:, m, :], 0).T, cmap="autumn", alpha=0.4, origin="lower")
    plt.axis("off")
plt.show()

**You should see** four pictures: the prediction (bottom row) should cover the same region as the true
mask (top row), in both views.

# STEP 8: Lung volume

With a 3D mask we can measure something a doctor understands: the volume of the lungs in litres.

The original voxels are 2.5 mm wide. After the resize to 96 x 96 x 96 they are larger: along each axis the size
becomes `2.5 * original size / 96`. The cell computes the volume of one voxel of the resized scan, in litres.

1. Compute `true_volume`: number of voxels equal to 1 in `label`, times `voxel_litres`.
2. Compute `pred_volume` in the same way from `pred`.
3. Print the two values.

In [ ]:
# Shape of the first validation scan before the resize
original_shape = nib.load(val_files[0]["image"]).shape

voxel_mm3 = (2.5 * original_shape[0] / IMG_SIZE) * (2.5 * original_shape[1] / IMG_SIZE) * (2.5 * original_shape[2] / IMG_SIZE)
voxel_litres = voxel_mm3 / 1_000_000

# 1. Volume of the reference mask (hint: label.sum().item() * voxel_litres)
true_volume = ...

# 2. Volume of the prediction
pred_volume = ...

# 3. Print
print(...)
print(...)

**You should see** two similar values, between 2 and 8 litres: the size of the lungs of an adult holding
their breath.

# Questions to discuss

1. The 3D network sees voxels of about 4 mm, the 2D network pixels of about 1.5 mm. Which one draws the border
   of the lung more precisely? Which one knows more about what is above and below a slice?
2. We have one training example per patient in both cases. In which case is an example "richer"?
3. Why did we have to reduce the batch size?

# To go further

- MONAI tutorial, 3D spleen segmentation. It introduces the two techniques used with full-resolution scans,
  training on random patches and sliding-window inference: https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/spleen_segmentation_3d.ipynb
- nnU-Net, the method that configures itself for a new dataset: https://arxiv.org/abs/1809.10486
- TotalSegmentator, a ready model for more than 100 structures in CT: https://arxiv.org/abs/2208.05868
- Lung nodules, a harder segmentation problem: notebook 04.